Setup

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import os
import numpy as np
import pandas as pd
from pathlib import Path
import sys

sys.path.append(os.path.join(Path.cwd(), "VTFW_Lab4"))

from VTFW_Lab4.data.data_writer import get_data_dir
from VTFW_Lab4.data.data_writer import get_data_dir, write_flying_wing
from VTFW_Lab4.geometry.naca_airfoil import NACA4, NACA5
from VTFW_Lab4.geometry.generators import get_wing_geometries, create_trapezoidal_dimensionalized_wing
from VTFW_Lab4.data.data_classes import FlyingWing
from VTFW_Lab4.data.data_reader import read_flying_wing, read_list_of_flying_wings
from VTFW_Lab4.utils.common import find_simulation
from VTFW_Lab4.utils.common import find_analysis
from VTFW_Lab4.geometry.wing_geometry import get_trapezoidal_aspect_ratio, get_taper_ratio, get_sweep_angle
from VTFW_Lab4.utils.aerosandbox_interface import run_asb_vlm
from VTFW_Lab4.analysis.cg_boundary_analysis import get_cg_for_trim
from VTFW_Lab4.analysis.cg_boundary_analysis import get_cm_cg
# Initialise data directory for data storage 

data_dir_name = f"Bachelor_Thesis_Data_Dir_Validation"

vlm_simulation_id = "bat_vlm_simu_1"
vlm_analysis_id = "bat_vlm_analysis_3"


data_dir = get_data_dir(data_dir_name, os.path.join(Path.cwd(), "data"))

#os.startfile(data_dir)


Wing geometries

In [ ]:
good_flying_wing = FlyingWing("good_fw",  create_trapezoidal_dimensionalized_wing(1.0, 10, 30, NACA4("1415")), [], [])

bad_flying_wing = FlyingWing("bad_fw", create_trapezoidal_dimensionalized_wing(0.2, 4, 0, NACA4("4515")), [], [])

flying_wings = [good_flying_wing, bad_flying_wing]

prev_flying_wings = read_list_of_flying_wings(data_dir)

for flying_wing in flying_wings:
    #if not any(flying_wing.id == prev_flying_wing.id for prev_flying_wing in prev_flying_wings):
        write_flying_wing(data_dir, flying_wing)


VLM Simulation

In [ ]:
from VTFW_Lab4.utils.aerosandbox_interface import run_multi_asb_vlm

alpha_values = np.linspace(0, 10, 11)
twist_values = np.linspace(-5, 5, 11)

flying_wings = read_list_of_flying_wings(data_dir)

flying_wings = list(filter(lambda flying_wing: len(find_simulation(flying_wing, vlm_simulation_id)) == 0, flying_wings))

for flying_wing in tqdm(flying_wings, desc="Processing simulations", unit="iteration"):
    for twist in twist_values:

        flying_wing.wing_geometry.current_total_twist = twist

        run_multi_asb_vlm(flying_wing, vlm_simulation_id, [100], alpha_values, [0], {}, 2, 10, 10)


    write_flying_wing(data_dir, flying_wing)

Analysis

In [ ]:
from VTFW_Lab4.analysis.cg_boundary_analysis import cg_boundary_twist_analysis

flying_wings = read_list_of_flying_wings(data_dir)
#flying_wings = list(filter(lambda flying_wing: len(find_analysis(flying_wing, vlm_analysis_id)) == 0, flying_wings))

for flying_wing in tqdm(flying_wings, desc="Processing analysis", unit="iteration"):

    cg_boundary_twist_analysis(flying_wing, vlm_simulation_id, vlm_analysis_id)
    write_flying_wing(data_dir, flying_wing)


In [ ]:
from VTFW_Lab4.utils.common import print_data_dict

flying_wings = read_list_of_flying_wings(data_dir)

flying_wings[0].cg = 0.4415
flying_wings[1].cg = 1.662

for flying_wing in flying_wings:

    vlm_analysis = find_analysis(flying_wing, vlm_analysis_id)[-1]

    print_data_dict(vlm_analysis.output_data)

    alpha_values = vlm_analysis.output_data["Alpha values"]
    plt.plot(alpha_values, vlm_analysis.output_data["Lower CG boundary"], "-o", color = "red", label = "Upper and lower boundary")
    plt.plot(alpha_values, vlm_analysis.output_data["Upper CG boundary"], "-o", color = "red")

    plt.plot(alpha_values, np.ones_like(alpha_values)*vlm_analysis.output_data["Upper CG limit"], "--", color = "green", label= "Upper and lower constant limit")
    plt.plot(alpha_values, np.ones_like(alpha_values)*vlm_analysis.output_data["Lower CG limit"], "--", color = "green")

    plt.plot(alpha_values, np.ones_like(alpha_values)*flying_wing.cg, "--", color = "blue", label = "CG postion")

    plt.xlabel("Angle of atack")
    plt.ylabel("X Corrdinate")
    plt.title("CG postion limit analysis")
    plt.legend()
    plt.grid(True)
    plt.show()
